### Imports

In [1]:
import os
import cv2
import numpy as np
%matplotlib inline
from utils import *
import pandas as pd
import seaborn as sns
from modules import *
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import utils.for_eye_tracker as et
from scipy.signal import savgol_filter

Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/heartpy/datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


________

## Get gaze data ready

In [2]:
def load_gaze(csv_path):
    df = pd.read_csv(csv_path)

    # Rename for consistency
    df = df.rename(columns={
        'Seconds': 'time',
        'GazeX': 'x',
        'GazeY': 'y'
    })

    # Convert time
    df['time'] = pd.to_datetime(df['time'])

    return df[['time', 'x', 'y']]

In [42]:
gaze_df = load_gaze("/mnt/raid/emotional_data_raquel/supp_mine/gaze/sub-OE002/ses-Hellerup/gaze.csv")
gaze_df.head()

,time,x,y
0,2024-06-27 06:49:20.044713708,684.03870,488.66174
1,2024-06-27 06:49:20.074729708,692.71780,422.27580
2,2024-06-27 06:49:20.080713708,629.63460,352.35130
3,2024-06-27 06:49:20.087721708,582.17580,372.18850
4,2024-06-27 06:49:20.122729708,374.60846,394.04465


In [4]:
gaze_df['dt'] = gaze_df['time'].diff().dt.total_seconds()
print(gaze_df['dt'].describe())

count    44661.000000
mean         0.033314
std          0.028478
min          0.000000
25%          0.020000
50%          0.031008
75%          0.036992
max          0.436000
Name: dt, dtype: float64


min being $0.000 s$ indicates that there are duplicated timestamps
and max being $0.436 s$ means that there is occasional gaps between observations

In [5]:
# Printing duplicated time rows
dup = gaze_df[gaze_df.duplicated(subset='time', keep=False)]
print(dup)

                               time          x         y        dt
1592  2024-06-27 06:50:13.085737708  463.89682  862.9679  0.011008
1593  2024-06-27 06:50:13.085737708  463.89682  862.9679  0.000000
1594  2024-06-27 06:50:13.085737708  463.89682  862.9679  0.000000
44655 2024-06-27 07:14:07.864713708  780.51560  825.3645  0.016000
44656 2024-06-27 07:14:07.864713708  780.51560  825.3645  0.000000
44657 2024-06-27 07:14:07.864713708  780.51560  825.3645  0.000000
44658 2024-06-27 07:14:07.864713708  780.51560  825.3645  0.000000
44659 2024-06-27 07:14:07.864713708  780.51560  825.3645  0.000000
44660 2024-06-27 07:14:07.864713708  780.51560  825.3645  0.000000
44661 2024-06-27 07:14:07.864713708  780.51560  825.3645  0.000000


In [6]:
# Deleting duplicated rows based on above
gaze_df = gaze_df.drop_duplicates(subset='time')

# Check stats again to ensure min value changed
gaze_df['dt'] = gaze_df['time'].diff().dt.total_seconds()
print(gaze_df['dt'].describe())

count    44653.000000
mean         0.033320
std          0.028477
min          0.004992
25%          0.020000
50%          0.031008
75%          0.036992
max          0.436000
Name: dt, dtype: float64


In [7]:
print(gaze_df[['x','y']].describe())

                  x             y
count  44654.000000  44654.000000
mean     413.907478    443.224356
std      202.171015    193.481324
min     -164.080370     52.209503
25%      303.201225    288.008457
50%      416.337830    391.041080
75%      521.257462    567.746005
max     1045.218000   1008.953600


In [8]:
gaze_df = gaze_df[(gaze_df['x'] >= 0) & (gaze_df['y'] >= 0)]
print(gaze_df[['x','y']].describe())

                  x             y
count  43272.000000  43272.000000
mean     428.636696    444.004630
std      187.444502    195.343474
min        0.114891     52.209503
25%      317.742110    286.723085
50%      421.289965    391.613825
75%      526.775720    571.591437
max     1045.218000   1008.953600


In [9]:
gaze_df.head()

,time,x,y,dt
0,2024-06-27 06:49:20.044713708,684.03870,488.66174,NaN
1,2024-06-27 06:49:20.074729708,692.71780,422.27580,0.030016
2,2024-06-27 06:49:20.080713708,629.63460,352.35130,0.005984
3,2024-06-27 06:49:20.087721708,582.17580,372.18850,0.007008
4,2024-06-27 06:49:20.122729708,374.60846,394.04465,0.035008


Now with no duplicates and good quality data we actually begin the eye tracking part.

_______

## Smooth and remove noise from gaze coordinates

Here i tried to do $savgol_filter$ with $window_length$ of first 11, then 2, then 3, then finally 5.

With 11 the result in the columns was way too modified and therefore I wanted to keep this value smaller. Value 2 is not possible since $polyorder$ needs to be bigger than $window_length$, therefore I then tried value of 3, but that just keep the results the exact same as the "x" and "y" columns. Since the goal here is to smooth the data and remove its noise (and the $window_length$ vakue needs to be odd), I ended up with the value of 5, not too much smoothing (11) and also not no smoothing a tall (3), a good in between.

In [10]:
def preprocess_gaze(df, x_col='x', y_col='y', time_col='time'):
    df = df.copy()

    # Sort by time
    df = df.sort_values(time_col)

    # Interpolate missing values
    df[x_col] = df[x_col].interpolate(limit=5)
    df[y_col] = df[y_col].interpolate(limit=5)

    # Remove remaining NaNs
    df = df.dropna(subset=[x_col, y_col])

    # Smoothing data with Savitzky-Golay
    if len(df) > 5:
        df['x_smooth'] = savgol_filter(df[x_col], 5, 2)
        df['y_smooth'] = savgol_filter(df[y_col], 5, 2)
    else:
        df['x_smooth'] = df[x_col]
        df['y_smooth'] = df[y_col]

    return df

In [11]:
gaze = preprocess_gaze(gaze_df)
gaze.head()

,time,x,y,dt,x_smooth,y_smooth
0,2024-06-27 06:49:20.044713708,684.03870,488.66174,NaN,678.962928,491.803820
1,2024-06-27 06:49:20.074729708,692.71780,422.27580,0.030016,695.351640,410.818983
2,2024-06-27 06:49:20.080713708,629.63460,352.35130,0.005984,652.187712,367.869272
3,2024-06-27 06:49:20.087721708,582.17580,372.18850,0.007008,538.997040,368.672943
4,2024-06-27 06:49:20.122729708,374.60846,394.04465,0.035008,413.674246,386.541165


Making sure no important data is left behind

In [12]:
plt.figure(figsize=(12,4))
plt.plot(gaze['x'], alpha=0.3, label='raw')
plt.plot(gaze['x_smooth'], label='smooth (window=5)')
plt.legend()
plt.title("Raw vs Smoothed Gaze")
plt.show()

## Compute gaze velocity (so I-VT ready)

In [13]:
def compute_velocity(df, time_col='time'):
    df = df.copy()

    # Convert time to seconds
    t_sec = pd.to_datetime(df[time_col]).astype('int64') / 1e9

    # Compute differences
    dt = np.diff(t_sec)
    dx = np.diff(df['x_smooth'])
    dy = np.diff(df['y_smooth'])

    # Avoid division by zero
    dt[dt == 0] = np.nan

    # Compute velocity
    velocity = np.sqrt(dx**2 + dy**2) / dt

    # Add to dataframe
    df['velocity'] = np.nan
    df.iloc[1:, df.columns.get_loc('velocity')] = velocity

    return df

In [14]:
gaze = compute_velocity(gaze)
gaze.head()

,time,x,y,dt,x_smooth,y_smooth,velocity
0,2024-06-27 06:49:20.044713708,684.03870,488.66174,NaN,678.962928,491.803820,NaN
1,2024-06-27 06:49:20.074729708,692.71780,422.27580,0.030016,695.351640,410.818983,2752.752557
2,2024-06-27 06:49:20.080713708,629.63460,352.35130,0.005984,652.187712,367.869272,10175.640925
3,2024-06-27 06:49:20.087721708,582.17580,372.18850,0.007008,538.997040,368.672943,16151.869596
4,2024-06-27 06:49:20.122729708,374.60846,394.04465,0.035008,413.674246,386.541165,3616.040827


In [15]:
gaze['velocity'].describe()

count    43271.000000
mean      1449.715392
std       2290.396923
min          0.000000
25%        238.135928
50%        625.050659
75%       1688.926877
max      46470.783386
Name: velocity, dtype: float64

Since the data here is the participants gaze mapped onto video frame coordinates, therefore the measure here is pixels per second.


In [16]:
plt.hist(gaze['velocity'].dropna(), bins=100)
plt.title("Velocity distribution")
plt.show()

## Implement Identification by Velocity Threshold (I-VT) algorithm 

The threshold chosen here to do the I-VT classification is 1000 since its a good in between value between the median (≈ 630) and 75% (≈ 1700) of the $velocity$ parameter. So it sits nicely between the fixation cluster as well as the saccade cluster:

fixations      | threshold | saccades </p>
----peak-------|---1000----|------tail-----

In [17]:
def classify_ivt(df, threshold=1000):
    df = df.copy()

    df['event'] = 'fixation'
    df.loc[df['velocity'] > threshold, 'event'] = 'saccade'

    return df

In [18]:
gaze = classify_ivt(gaze, threshold=1000)
gaze['event'].value_counts()

event
fixation    27085
saccade     16187
Name: count, dtype: int64

In [19]:
plt.figure(figsize=(12,4))
plt.plot(gaze['velocity'], alpha=0.5)
plt.axhline(1000, color='red', linestyle='--', label='threshold')
plt.legend()
plt.title("Velocity with threshold")
plt.show()

In [20]:
gaze

,time,x,y,dt,x_smooth,y_smooth,velocity,event
0,2024-06-27 06:49:20.044713708,684.03870,488.66174,NaN,678.962928,491.803820,NaN,fixation
1,2024-06-27 06:49:20.074729708,692.71780,422.27580,0.030016,695.351640,410.818983,2752.752557,saccade
2,2024-06-27 06:49:20.080713708,629.63460,352.35130,0.005984,652.187712,367.869272,10175.640925,saccade
3,2024-06-27 06:49:20.087721708,582.17580,372.18850,0.007008,538.997040,368.672943,16151.869596,saccade
4,2024-06-27 06:49:20.122729708,374.60846,394.04465,0.035008,413.674246,386.541165,3616.040827,saccade
...,...,...,...,...,...,...,...,...
44651,2024-06-27 07:14:07.747721708,469.23538,608.26580,0.016992,472.646924,614.141474,1015.513036,saccade
44652,2024-06-27 07:14:07.808713708,492.12430,643.80115,0.060992,495.264811,630.322414,455.960092,fixation
44653,2024-06-27 07:14:07.820713708,532.62990,650.76807,0.012000,505.701261,665.095100,3025.402550,saccade
44654,2024-06-27 07:14:07.848713708,540.71643,732.54570,0.028000,601.488048,730.916267,4150.800962,saccade


________

# Focusing on fixations

In [21]:
def extract_fixations(df):
    df = df.copy()

    fixations = []
    current_fix = None

    for i, row in df.iterrows():
        if row['event'] == 'fixation':
            if current_fix is None:
                current_fix = {
                    'start_time': row['time'],
                    'x': [],
                    'y': []
                }

            # collects ALL gaze points during a fixation
            current_fix['x'].append(row['x'])
            current_fix['y'].append(row['y'])

        else:
            if current_fix is not None:
                end_time = df.loc[i-1, 'time']

                fixations.append({
                    'start_time': current_fix['start_time'],
                    'end_time': end_time,
                    'duration': (end_time - current_fix['start_time']).total_seconds(),
                    # computes the center of a fixation
                    'x_mean': np.mean(current_fix['x']),
                    'y_mean': np.mean(current_fix['y'])
                })

                current_fix = None

    return pd.DataFrame(fixations)

In [22]:
gaze = gaze.reset_index(drop=True)
fixations = extract_fixations(gaze)
fixations

,start_time,end_time,duration,x_mean,y_mean
0,2024-06-27 06:49:20.044713708,2024-06-27 06:49:20.044713708,0.000000,684.038700,488.661740
1,2024-06-27 06:49:20.190729708,2024-06-27 06:49:20.316713708,0.125984,340.015138,337.364814
2,2024-06-27 06:49:20.419721708,2024-06-27 06:49:20.419721708,0.000000,376.142700,389.921050
3,2024-06-27 06:49:20.686729708,2024-06-27 06:49:20.686729708,0.000000,296.711000,595.981930
4,2024-06-27 06:49:20.786729708,2024-06-27 06:49:20.859721708,0.072992,450.663153,491.086577
...,...,...,...,...,...
5360,2024-06-27 07:14:06.750729708,2024-06-27 07:14:06.929737708,0.179008,258.452292,252.398587
5361,2024-06-27 07:14:07.283721708,2024-06-27 07:14:07.283721708,0.000000,612.400300,537.299260
5362,2024-06-27 07:14:07.593737708,2024-06-27 07:14:07.593737708,0.000000,533.664300,589.195860
5363,2024-06-27 07:14:07.653737708,2024-06-27 07:14:07.717737708,0.064000,502.072860,602.705900


### Convert duration parameter from seconds to milliseconds

In [23]:
# convert from seconds to milliseconds
fixations['duration_ms'] = fixations['duration'] * 1000
# remove short fixations
fixations = fixations[fixations['duration_ms'] >= 100]
fixations

,start_time,end_time,duration,x_mean,y_mean,duration_ms
1,2024-06-27 06:49:20.190729708,2024-06-27 06:49:20.316713708,0.125984,340.015138,337.364814,125.984
8,2024-06-27 06:49:21.456713708,2024-06-27 06:49:21.588713708,0.132000,110.137752,313.486890,132.000
11,2024-06-27 06:49:21.923721708,2024-06-27 06:49:22.218729708,0.295008,218.694784,419.854572,295.008
12,2024-06-27 06:49:22.519721708,2024-06-27 06:49:22.714729708,0.195008,260.338998,309.797797,195.008
14,2024-06-27 06:49:22.894729708,2024-06-27 06:49:23.347721708,0.452992,68.475335,330.539858,452.992
...,...,...,...,...,...,...
5350,2024-06-27 07:14:03.056713708,2024-06-27 07:14:03.917737708,0.861024,617.541214,786.267597,861.024
5355,2024-06-27 07:14:04.783721708,2024-06-27 07:14:05.026729708,0.243008,827.314645,760.062724,243.008
5358,2024-06-27 07:14:05.584713708,2024-06-27 07:14:05.950729708,0.366016,390.385072,259.650397,366.016
5359,2024-06-27 07:14:06.386729708,2024-06-27 07:14:06.615721708,0.228992,94.397087,260.316978,228.992


In [24]:
fixations['duration_ms'].describe()

count     1789.000000
mean       380.595868
std        600.607080
min        100.000000
25%        160.000000
50%        229.984000
75%        386.016000
max      10198.976000
Name: duration_ms, dtype: float64

### Delete outliers

max now is $10198.976000$ which might be due to a stop or something so I will keep fixations from 100-400 ms max as research mentions.
Although this should not be done, I will keep this 400ms rule to rule out outliners. Like we can see this dataset what a 10198 ms outlier and this rule deletes it :)

In [25]:
fixations = fixations[fixations['duration_ms'] < 4000]
print(fixations['duration_ms'].describe())

count    1777.000000
mean      344.116488
std       382.709891
min       100.000000
25%       160.000000
50%       228.992000
75%       374.976000
max      3831.008000
Name: duration_ms, dtype: float64


In [26]:
fixations

,start_time,end_time,duration,x_mean,y_mean,duration_ms
1,2024-06-27 06:49:20.190729708,2024-06-27 06:49:20.316713708,0.125984,340.015138,337.364814,125.984
8,2024-06-27 06:49:21.456713708,2024-06-27 06:49:21.588713708,0.132000,110.137752,313.486890,132.000
11,2024-06-27 06:49:21.923721708,2024-06-27 06:49:22.218729708,0.295008,218.694784,419.854572,295.008
12,2024-06-27 06:49:22.519721708,2024-06-27 06:49:22.714729708,0.195008,260.338998,309.797797,195.008
14,2024-06-27 06:49:22.894729708,2024-06-27 06:49:23.347721708,0.452992,68.475335,330.539858,452.992
...,...,...,...,...,...,...
5350,2024-06-27 07:14:03.056713708,2024-06-27 07:14:03.917737708,0.861024,617.541214,786.267597,861.024
5355,2024-06-27 07:14:04.783721708,2024-06-27 07:14:05.026729708,0.243008,827.314645,760.062724,243.008
5358,2024-06-27 07:14:05.584713708,2024-06-27 07:14:05.950729708,0.366016,390.385072,259.650397,366.016
5359,2024-06-27 07:14:06.386729708,2024-06-27 07:14:06.615721708,0.228992,94.397087,260.316978,228.992


In [27]:
sns.histplot(fixations['duration_ms'], bins=50)
plt.title("Fixation Duration Distribution")
plt.show()

In [28]:
plt.figure(figsize=(6,6))
plt.scatter(fixations['x_mean'], fixations['y_mean'], s=2)
plt.title("Fixation Locations")
plt.gca().invert_yaxis()
plt.show()

In [29]:
# Get fixation center time to merge with video
fixations["mid_time"] = fixations["start_time"] + (fixations["end_time"] - fixations["start_time"]) / 2
fixations

,start_time,end_time,duration,x_mean,y_mean,duration_ms,mid_time
1,2024-06-27 06:49:20.190729708,2024-06-27 06:49:20.316713708,0.125984,340.015138,337.364814,125.984,2024-06-27 06:49:20.253721708
8,2024-06-27 06:49:21.456713708,2024-06-27 06:49:21.588713708,0.132000,110.137752,313.486890,132.000,2024-06-27 06:49:21.522713708
11,2024-06-27 06:49:21.923721708,2024-06-27 06:49:22.218729708,0.295008,218.694784,419.854572,295.008,2024-06-27 06:49:22.071225708
12,2024-06-27 06:49:22.519721708,2024-06-27 06:49:22.714729708,0.195008,260.338998,309.797797,195.008,2024-06-27 06:49:22.617225708
14,2024-06-27 06:49:22.894729708,2024-06-27 06:49:23.347721708,0.452992,68.475335,330.539858,452.992,2024-06-27 06:49:23.121225708
...,...,...,...,...,...,...,...
5350,2024-06-27 07:14:03.056713708,2024-06-27 07:14:03.917737708,0.861024,617.541214,786.267597,861.024,2024-06-27 07:14:03.487225708
5355,2024-06-27 07:14:04.783721708,2024-06-27 07:14:05.026729708,0.243008,827.314645,760.062724,243.008,2024-06-27 07:14:04.905225708
5358,2024-06-27 07:14:05.584713708,2024-06-27 07:14:05.950729708,0.366016,390.385072,259.650397,366.016,2024-06-27 07:14:05.767721708
5359,2024-06-27 07:14:06.386729708,2024-06-27 07:14:06.615721708,0.228992,94.397087,260.316978,228.992,2024-06-27 07:14:06.501225708


___________

In [30]:
participant_path = "/mnt/raid/emotional_data_raquel/data/OE002"
session_folder = os.listdir(participant_path)[0]
session_path = os.path.join(participant_path, session_folder)

datapicker = create_datapicker(path=session_path, schema=build_schema)
dataset = load_dataset(datapicker.selected_path, schema=build_schema)

gaze = dataset.streams.PupilLabs.PupilGaze.data

# rename for convenience and remove duplicates timestamps with mean coordinates
gaze = (
    gaze.groupby(level=0)[["GazeX", "GazeY"]]
    .mean()
    .rename(columns={"GazeX": "x", "GazeY": "y"})
)

print(gaze.head())
print(gaze.columns)
print(gaze.index.is_unique)

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
                                        x           y
Seconds                                              
2024-06-27 06:49:18.880713708  650.312439  593.095764
2024-06-27 06:49:18.935721708  647.516846  594.024963
2024-06-27 06:49:18.936713708  641.837463  611.282593
2024-06-27 06:49:18.938729708  642.202515  615.281128
2024-06-27 06:49:18.942729708  644.887878  615.760498
Index(['x', 'y'], dtype='object')
True


In [31]:
decoded_frames = dataset.streams.PupilLabs.DecodedFrames.data

# keep valid frames only
decoded_frames = decoded_frames[decoded_frames.Value > 0]

gaze = gaze.reindex(decoded_frames.index, method="nearest")

# assign frame numbers
gaze["frame"] = np.arange(len(gaze))

In [32]:
print(gaze.head())
print(gaze.tail())

                                        x           y  frame
Seconds                                                     
2024-06-27 06:49:20.044713708  639.500000  496.055115      0
2024-06-27 06:49:20.074729708  692.717773  422.275787      1
2024-06-27 06:49:20.080713708  629.634583  352.351288      2
2024-06-27 06:49:20.087721708  582.175781  372.188507      3
2024-06-27 06:49:20.122729708  374.608459  394.044647      4
                                        x           y  frame
Seconds                                                     
2024-06-27 07:14:07.864713708  630.668152  796.128357  44657
2024-06-27 07:14:07.864713708  630.668152  796.128357  44658
2024-06-27 07:14:07.864713708  630.668152  796.128357  44659
2024-06-27 07:14:07.864713708  630.668152  796.128357  44660
2024-06-27 07:14:07.864713708  630.668152  796.128357  44661


In [33]:
aligned = pd.merge_asof(
    fixations.sort_values("mid_time"),
    gaze,
    left_on="mid_time",
    right_index=True,
    direction="nearest"
)
aligned

,start_time,end_time,duration,x_mean,y_mean,duration_ms,mid_time,x,y,frame
1,2024-06-27 06:49:20.190729708,2024-06-27 06:49:20.316713708,0.125984,340.015138,337.364814,125.984,2024-06-27 06:49:20.253721708,365.590851,332.120728,8
8,2024-06-27 06:49:21.456713708,2024-06-27 06:49:21.588713708,0.132000,110.137752,313.486890,132.000,2024-06-27 06:49:21.522713708,112.952431,311.513092,46
11,2024-06-27 06:49:21.923721708,2024-06-27 06:49:22.218729708,0.295008,218.694784,419.854572,295.008,2024-06-27 06:49:22.071225708,213.662292,423.018829,63
12,2024-06-27 06:49:22.519721708,2024-06-27 06:49:22.714729708,0.195008,260.338998,309.797797,195.008,2024-06-27 06:49:22.617225708,264.875153,307.320892,79
14,2024-06-27 06:49:22.894729708,2024-06-27 06:49:23.347721708,0.452992,68.475335,330.539858,452.992,2024-06-27 06:49:23.121225708,67.573761,330.491943,94
...,...,...,...,...,...,...,...,...,...,...
5350,2024-06-27 07:14:03.056713708,2024-06-27 07:14:03.917737708,0.861024,617.541214,786.267597,861.024,2024-06-27 07:14:03.487225708,611.788330,791.358215,44523
5355,2024-06-27 07:14:04.783721708,2024-06-27 07:14:05.026729708,0.243008,827.314645,760.062724,243.008,2024-06-27 07:14:04.905225708,816.225769,765.305542,44566
5358,2024-06-27 07:14:05.584713708,2024-06-27 07:14:05.950729708,0.366016,390.385072,259.650397,366.016,2024-06-27 07:14:05.767721708,402.256531,269.071259,44592
5359,2024-06-27 07:14:06.386729708,2024-06-27 07:14:06.615721708,0.228992,94.397087,260.316978,228.992,2024-06-27 07:14:06.501225708,91.438522,258.237152,44613


In [34]:
video_path = "/mnt/raid/emotional_data_raquel/data/OE002/Copenhagen_Hellerup_sub-OE204002_2024-06-27T064915Z/pupil_video.avi"
cap = cv.VideoCapture(video_path)

In [35]:
ret, frame = cap.read()

gaze_rows = aligned[aligned["frame"] == 0]

for _, gaze in gaze_rows.iterrows():
    x = int(gaze["x"])
    y = int(gaze["y"])
    cv.circle(frame, (x, y), 10, (0, 0, 255), -1)

plt.imshow(cv.cvtColor(frame, cv.COLOR_BGR2RGB))
plt.title("Frame with gaze")
plt.axis("off")
plt.show()

In [36]:
frame_id = int(aligned["frame"].iloc[1])

cap.set(cv.CAP_PROP_POS_FRAMES, frame_id)
ret, frame = cap.read()

gaze_rows = aligned[aligned["frame"] == frame_id]

for _, gaze in gaze_rows.iterrows():
    x = int(gaze["x"])
    y = int(gaze["y"])
    cv.circle(frame, (x, y), 10, (0, 0, 255), -1)

plt.imshow(cv.cvtColor(frame, cv.COLOR_BGR2RGB))
plt.title(f"Frame {frame_id}")
plt.axis("off")
plt.show()

In [37]:
patch_size = 100

x = int(gaze_rows.iloc[0]["x"])
y = int(gaze_rows.iloc[0]["y"])

crop = frame[
    max(0, y-patch_size):y+patch_size,
    max(0, x-patch_size):x+patch_size
]

plt.imshow(cv.cvtColor(crop, cv.COLOR_BGR2RGB))
plt.title("What the participant sees")
plt.axis("off")
plt.show()

In [38]:
# path to COCO annotations file
ann_file = "/home/s243636/master-thesis/notebooks/instances_val2017.json"

coco = COCO(ann_file)

# get category names
cats = coco.loadCats(coco.getCatIds())
#labels = [cat["name"] for cat in cats]
labels = [f"a photo of a {c['name']}" for c in cats]

print(labels[:10])
print(len(labels))

loading annotations into memory...
Done (t=0.60s)
creating index...
index created!
['a photo of a person', 'a photo of a bicycle', 'a photo of a car', 'a photo of a motorcycle', 'a photo of a airplane', 'a photo of a bus', 'a photo of a train', 'a photo of a truck', 'a photo of a boat', 'a photo of a traffic light']
80
